# Expert Patch Embeddings from AlphaEarth Rasters

Converts a directory of embedding-vector GeoTiffs into "expert" patch embeddings.

An expert embedding assembles the mean, std, min, and max of the embedding values across a patch and concatenates them into a single long vector (4 × embedding_dim features per patch).

The notebook outputs a GeoDataFrame where each row contains one expert embedding and the centroid point geometry of its patch, saved as a GeoPackage.

## 1. Imports

In [ ]:
from glob import glob
from pathlib import Path
import sys
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

# Resolve the project root so `tile_utils` can be imported from the sibling
# `gee` directory, whether this runs as a notebook or a script.
try:
    _root = Path(__file__).resolve().parent.parent
except NameError:
    _root = Path.cwd()
    if not (_root / "gee" / "tile_utils.py").exists() and (_root.parent / "gee" / "tile_utils.py").exists():
        _root = _root.parent
sys.path.insert(0, str(_root / "gee"))
from tile_utils import cut_chips

## 2. Input Rasters

Point `raster_embeddings_path` to a collection of GeoTiffs of embedding vectors, e.g. as downloaded from Earth Engine with `0_gee_data_pull.py`.

In [ ]:
raster_embeddings_path = Path('../data/neg_label_randAlphaEarth_cmb_160/')
paths = sorted(glob(str(raster_embeddings_path / '*.tif')))
if not paths:
    raise FileNotFoundError("No .tif files found. Set raster_embeddings_path to your embeddings directory.")

## 3. Expert Statistics

Summary statistics computed per patch and per embedding channel. It might be interesting to experiment with different sets of statistics here.

In [ ]:
def expert_stats(patches, axis=(2,3)): 
    """Compute statistics for expert embeddings. 
    
    patches: ndarray of shape (N, C, H, W)
    axis: tuple: Axes over which to take summary statistics.
    """
    means = patches.mean(axis=axis)  
    stds = patches.std(axis=axis)
    maxes = patches.max(axis=axis)
    mins = patches.min(axis=axis)
    return means, stds, maxes, mins

# Number of statistics per channel; sets the expert embedding width below.
n_stats = len(expert_stats(np.zeros((1,1,1,1))))
n_stats

## 4. Build Expert Embeddings

For each raster: cut it into patches, compute the per-patch statistics, flatten them into one vector per patch, and attach the patch centroid as the geometry.

In [ ]:
patch_size = 16   # Patch width/height in raster pixels.
stride = 1        # Step between consecutive patches.

# Embedding dimensionality (band count) taken from the first raster.
with rasterio.open(paths[0]) as f:
    embedding_dim = f.count  
    
feature_columns = [f"e{i}" for i in range(embedding_dim * n_stats)]
gdfs = []

for path in paths: 
    
    # Read the raster and its georeferencing.
    with rasterio.open(path) as f:
        bounds = f.bounds
        crs = f.crs
        img = f.read()
        
    assert img.shape[0] == embedding_dim
        
    # Cut into (N, H, W, C) patches, then move channels first for expert_stats.
    img = np.moveaxis(img, 0, -1)
    patches, geoms = cut_chips(img, bounds, chip_size=patch_size, stride=stride, crs=crs)
    patches = np.moveaxis(patches, -1, 1)
    
    # Stack the statistics into a single (N, C * n_stats) matrix.
    expert = np.array(expert_stats(patches))
    expert = np.moveaxis(expert, 0, -1)
    N, C, S = expert.shape
    expert = expert.reshape(N, C * S)
        
    # One row per patch: expert features + patch centroid geometry.
    df = pd.DataFrame(expert, columns=feature_columns)
    gdf = gpd.GeoDataFrame(df, geometry=geoms['geometry'])
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, message="Geometry is in a geographic CRS")
        gdf['geometry'] = gdf.geometry.centroid
    gdfs.append(gdf)
    
combined_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    geometry="geometry",
    crs=crs
)

print(f"{len(combined_gdf)} expert embeddings created from {len(paths)} files")
combined_gdf.head()

## 5. Save

Write the expert embeddings next to the input rasters, with patch size and stride in the filename.

In [ ]:
combined_gdf.to_file(raster_embeddings_path / f"patch_size{patch_size}_stride{stride}_embeddings.gpkg", driver="GPKG", index=False)